# Bitacoras onomásticas SEMEFO 1968-1982
# Revisión de bitacoras

In [1]:
import re
import logging
import pandas as pd
from numpy import nan
from glob import glob
from os.path import exists
from ydata_profiling import ProfileReport
from helper_pandas import simplify_string, to_dtype_with_errors
from helper_pandas import join_strings_with_nans


/home/esther/anaconda3/envs/SEMEFO_busqueda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def check_bo(year, df):
    columns = {'ID':'categorical', 'Numero_progresivo':'int', 'Nombre_completo':'str', 'Primer_apellido':'str', 
               'Segundo_apellido':'str', 'Nombres_propios':'str', 'Fecha':'datetime', 'Expediente_Semefo':'categorical', 
               'Procedencia':'categorical', 'Numero_acta':'categorical', 'Diagnostico':'categorical', 'Sexo':'categorical', 
               'Edad':'int', 'Tipo_restos':'categorical', 'Bitacora_ingresos':'str', 'Pagina_PDF':'int', 'Foja':'str', 
               'Posible_identificado_como':'str', 'Observaciones_capturista':'str', }
    # check col names
    for col in columns.keys():
        if col not in df.columns:
            logging.warning('\t{}\t Missing col {}'.format(year,col))
    for col, type in columns.items():
        # format str columns
        if type=='str':
            df[col] = df[col].apply(simplify_string)
        # create columns with dtype
        elif type=='int':
            df = to_dtype_with_errors(df, col, dtype='numeric', downcast='integer')
        elif type=='datetime':
            df = to_dtype_with_errors(df, col, dtype='datetime')
    return df

In [3]:
def minimal_profile(df, f_profile, force_profile=False):
    if force_profile or not exists(f_profile):
        logging.info('\tProfile: {}'.format(f_profile))
        prof = df.profile_report(title=f_profile, minimal=True, correlations=None)
        prof.to_file(output_file=f_profile)

## Cargar df y convertir a tipo de datos

In [4]:
data_dir = 'data_raw/Transcritas_20230920/'
dfs = dict()

size = dict()
for file in glob(data_dir+'*'):
    # Load
    logging.warning(' Load: '+file)
    year = re.findall(r"\d+", file)[-1]
    df = pd.read_excel(file, dtype='str')
    size[file] = df.shape[0]
    logging.info('\tSize: {}'.format(df.shape))
    # Check
    df = check_bo(year, df)
    # Profile
    f_profile = file.split('/')[-1]
    f_profile = 'profiles/'+f_profile.replace('.xlsx', '_profile.html')
    minimal_profile(df, f_profile, force_profile=False)
    if size[file]!= df.shape[0]:
        logging.warning(f"Change in size of file {file}")
    # Save
    dfs[year] = df

df = pd.concat(dfs)
df = df.set_index('ID').sort_index()
df = df.dropna(how='all', axis=1)
df.to_pickle('data_raw/SEMEFO_BO_all_revision.pkl')
df

/home/esther/anaconda3/envs/SEMEFO_busqueda/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:312: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/home/esther/anaconda3/envs/SEMEFO_busqueda/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:312: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Numero_progresivo,Numero_progresivo_clean,Nombre_completo,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha,Fecha_clean,Fecha_error,Expediente_Semefo,...,Edad,Edad_clean,Edad_error,Tipo_restos,Bitacora_ingresos,Pagina_PDF,Pagina_PDF_clean,Foja,Posible_identificado_como,Observaciones_capturista
ID,,,,,,,,,,,,,,,,,,,,,
BO_1968_00001,S-D,NaN,acosta ortega teresa,acosta,ortega,teresa,1968-01-03,1968-01-03,NaN,37,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
BO_1968_00002,S-D,NaN,avila de cuestas catalina,avila,de cuestas,catalina,1968-01-05,1968-01-05,NaN,58,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
BO_1968_00003,S-D,NaN,arzate paredes juan,arzate,paredes,juan,1968-01-07,1968-01-07,NaN,83,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
BO_1968_00004,S-D,NaN,alvarez martinez isaac,alvarez,martinez,isaac,1968-01-07,1968-01-07,NaN,86,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
BO_1968_00005,S-D,NaN,arellano viuda de campos ma.,arellano,viuda de campos,ma.,1968-01-07,1968-01-07,NaN,88,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BO_1982_07489,S-D,NaN,placenta,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,3079,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
BO_1982_07490,S-D,NaN,5 dedos del pie derecho de desconocido,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,3060,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
BO_1982_07491,S-D,NaN,dedo de desconocido,s-d,s-d,s-d,1982-11-19,1982-11-19,NaN,6389,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d


# Procesar

In [5]:
df = pd.read_pickle('data_raw/SEMEFO_BO_all_revision.pkl')
df.reset_index(inplace=True)
df

,ID,Numero_progresivo,Numero_progresivo_clean,Nombre_completo,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha,Fecha_clean,Fecha_error,...,Edad,Edad_clean,Edad_error,Tipo_restos,Bitacora_ingresos,Pagina_PDF,Pagina_PDF_clean,Foja,Posible_identificado_como,Observaciones_capturista
0,BO_1968_00001,S-D,NaN,acosta ortega teresa,acosta,ortega,teresa,1968-01-03,1968-01-03,NaN,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
1,BO_1968_00002,S-D,NaN,avila de cuestas catalina,avila,de cuestas,catalina,1968-01-05,1968-01-05,NaN,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
2,BO_1968_00003,S-D,NaN,arzate paredes juan,arzate,paredes,juan,1968-01-07,1968-01-07,NaN,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
3,BO_1968_00004,S-D,NaN,alvarez martinez isaac,alvarez,martinez,isaac,1968-01-07,1968-01-07,NaN,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
4,BO_1968_00005,S-D,NaN,arellano viuda de campos ma.,arellano,viuda de campos,ma.,1968-01-07,1968-01-07,NaN,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96855,BO_1982_07489,S-D,NaN,placenta,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96856,BO_1982_07490,S-D,NaN,5 dedos del pie derecho de desconocido,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96857,BO_1982_07491,S-D,NaN,dedo de desconocido,s-d,s-d,s-d,1982-11-19,1982-11-19,NaN,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96858,BO_1982_07492,S-D,NaN,4 dedos de desconocido,s-d,s-d,s-d,1982-11-28,1982-11-28,NaN,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d


## ID

In [6]:
# Format
df.loc[(df['ID'].str.startswith('BO')!=True), ['ID', 'Nombre_completo', 'Fecha']]

,ID,Nombre_completo,Fecha


In [7]:
# Format
data = pd.DataFrame([row.split('_') for row in df['ID'].to_list()], columns=['BO', 'year','ID'])
data.index = df['ID']
data['year'] = pd.to_numeric(data['year'], downcast='integer')
data['ID'] = pd.to_numeric(data['ID'], downcast='integer')
data

data.groupby('year').agg({
            'BO':'size',
            'ID':['max', pd.Series.nunique]
            })

BO    ID        
      size   max nunique
year                    
1968  4719  4719    4719
1969  5052  5052    5052
1970  5303  5303    5303
1971  5524  5524    5524
1972  5770  5770    5770
1973  6471  6471    6471
1974  5731  5731    5731
1975  6861  6861    6861
1976  6907  6907    6907
1977  7166  7166    7166
1978  7154  7154    7154
1979  7461  7461    7461
1980  7619  7619    7619
1981  7629  7629    7629
1982  7493  7493    7493

## Numero_progresivo

In [8]:
#df['Numero_progresivo'] = df['Numero_progresivo_clean']
df.drop('Numero_progresivo_clean', axis=1, inplace=True)

## Fecha

In [9]:
df.loc[df['Fecha_error'].notna(), ['ID', 'Fecha', 'Fecha_error', 'Observaciones_capturista']]

,ID,Fecha,Fecha_error,Observaciones_capturista
10937,BO_1970_01167,1970-04-31,1970-04-31,"en la bitacora original se anoto 31 de abril, ..."
12766,BO_1970_02996,1970-02-29,1970-02-29,"1970 no fue un ano bisiesto, sin embargo en la..."
13004,BO_1970_03234,1970-02-29,1970-02-29,"1970 no fue un ano bisiesto, sin embargo en la..."
15392,BO_1971_00319,1971-02-29,1971-02-29,"nota: (dispensa). 1971 no fue un ano bisiesto,..."
26130,BO_1972_05533,1972-02-30,1972-02-30,al parecer la bitacora ssi senala que se entre...
34846,BO_1974_02008,1974-02-29,1974-02-29,"1974 no fue un ano bisiesto, sin embargo en la..."
45886,BO_1976_00456,1976-04-31,1976-04-31,"la fecha indica un dia que no existe, se corro..."
46018,BO_1976_00588,1976-11-31,1976-11-31,"la fecha indica un dia que no existe, se corro..."
46720,BO_1976_01290,1976-06-31,1976-06-31,"la fecha indica un dia que no existe, se corro..."
47891,BO_1976_02461,1976-02-31,1976-02-31,"la fecha indica un dia que no existe, se corro..."


## Procedencia

In [10]:
df['Procedencia'] = df['Procedencia'].apply(simplify_string, str_case=None)
cat = pd.read_csv('catalogos/catalogo_procedencia.csv')
df = pd.merge(df, cat, on='Procedencia', how='left')
df.columns

df = df.rename(columns={'Equivalencia':'Procedencia_clean', 'Direccion':'Procedencia_direccion', 
                        'Alcaldia':'Procedencia_alcaldia'})
df = df[['ID', 'Numero_progresivo', 'Nombre_completo', 'Primer_apellido', 'Segundo_apellido', 'Nombres_propios', 
         'Fecha', 'Fecha_clean', 'Fecha_error', 'Expediente_Semefo', 'Procedencia', 'Procedencia_clean', 
         'Procedencia_direccion', 'Procedencia_alcaldia', 'Numero_acta', 'Diagnostico', 'Sexo', 'Edad', 
         'Edad_clean', 'Edad_error', 'Tipo_restos', 'Bitacora_ingresos', 'Pagina_PDF', 'Pagina_PDF_clean',
         'Foja', 'Posible_identificado_como', 'Observaciones_capturista']]

df['Procedencia_clean'] = df['Procedencia_clean'].astype("category")
df['Procedencia_alcaldia'] = df['Procedencia_alcaldia'].astype("category")
df

,ID,Numero_progresivo,Nombre_completo,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha,Fecha_clean,Fecha_error,Expediente_Semefo,...,Edad,Edad_clean,Edad_error,Tipo_restos,Bitacora_ingresos,Pagina_PDF,Pagina_PDF_clean,Foja,Posible_identificado_como,Observaciones_capturista
0,BO_1968_00001,S-D,acosta ortega teresa,acosta,ortega,teresa,1968-01-03,1968-01-03,NaN,37,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
1,BO_1968_00002,S-D,avila de cuestas catalina,avila,de cuestas,catalina,1968-01-05,1968-01-05,NaN,58,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
2,BO_1968_00003,S-D,arzate paredes juan,arzate,paredes,juan,1968-01-07,1968-01-07,NaN,83,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
3,BO_1968_00004,S-D,alvarez martinez isaac,alvarez,martinez,isaac,1968-01-07,1968-01-07,NaN,86,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
4,BO_1968_00005,S-D,arellano viuda de campos ma.,arellano,viuda de campos,ma.,1968-01-07,1968-01-07,NaN,88,...,S-D,NaN,NaN,Cadáver,semefo_df_bo_1968,2,2,1,s-d,s-d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96855,BO_1982_07489,S-D,placenta,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,3079,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96856,BO_1982_07490,S-D,5 dedos del pie derecho de desconocido,s-d,s-d,s-d,1982-06-05,1982-06-05,NaN,3060,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96857,BO_1982_07491,S-D,dedo de desconocido,s-d,s-d,s-d,1982-11-19,1982-11-19,NaN,6389,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d
96858,BO_1982_07492,S-D,4 dedos de desconocido,s-d,s-d,s-d,1982-11-28,1982-11-28,NaN,6528,...,S-D,NaN,NaN,Miembros,semefo_df_bo_1982,251,251,156,s-d,s-d


In [11]:
procedencias = df[['Procedencia', 'Procedencia_clean', 'Procedencia_direccion', 'Procedencia_alcaldia']]
procedencias = procedencias.value_counts(dropna=False)
procedencias.to_csv('data_raw/conteo_procedencia.csv')
procedencias

Procedencia  Procedencia_clean  Procedencia_direccion                        Procedencia_alcaldia
S-D          S-D                Sin datos                                    nan                     26862
32a          32A                Ministerio Público 32                        nan                      2073
33a          33A                Ministerio Público 33 (Col Jardín Balbuena)  Venustiano Carranza      1970
37a          37A                Ministerio Público 37 (Col Polanco)          Miguel Hidalgo           1701
32           32A                Ministerio Público 32                        nan                      1562
                                                                                                     ...  
C-H-6        nan                NaN                                          nan                         1
C-H-5        nan                NaN                                          nan                         1
C-H          nan                NaN           

## Numero_acta

In [12]:
data = df[['Procedencia', 'Numero_acta']].replace('S-D',nan) \
         .apply(join_strings_with_nans, sep=' : ', axis=1) \
         .fillna('S-D')
index = df.columns.get_loc('Numero_acta')
df.insert(index+1, 'Acta_procedencia', data)
df.columns

Index(['ID', 'Numero_progresivo', 'Nombre_completo', 'Primer_apellido',
       'Segundo_apellido', 'Nombres_propios', 'Fecha', 'Fecha_clean',
       'Fecha_error', 'Expediente_Semefo', 'Procedencia', 'Procedencia_clean',
       'Procedencia_direccion', 'Procedencia_alcaldia', 'Numero_acta',
       'Acta_procedencia', 'Diagnostico', 'Sexo', 'Edad', 'Edad_clean',
       'Edad_error', 'Tipo_restos', 'Bitacora_ingresos', 'Pagina_PDF',
       'Pagina_PDF_clean', 'Foja', 'Posible_identificado_como',
       'Observaciones_capturista'],
      dtype='object')

In [13]:
df['Acta_procedencia'].value_counts(dropna=False)

Acta_procedencia
S-D           25651
26a : 144        27
TP-1 : 868       13
10 : 1658        12
1a : 3737        11
              ...  
HRL : 423         1
VC-2 : 862        1
CH-1 : 422        1
HCM : 79          1
9a : 4193         1
Name: count, Length: 61826, dtype: int64

## Diagnostico

In [14]:
df.shape

(96860, 28)

In [15]:
cat = pd.read_csv('catalogos/catalogo_diagnostico_simplificado.csv')
df['Diagnostico'] = df['Diagnostico'].apply(simplify_string, str_case=None)
df = df.merge(cat, on='Diagnostico', how='left')
df = df.rename({'Equivalencia':'Diagnostico_estandar','Extendido':'Diagnostico_extendido'}, axis=1)
df = df[['ID', 'Numero_progresivo', 'Nombre_completo', 'Primer_apellido', 'Segundo_apellido', 'Nombres_propios', 
         'Fecha', 'Fecha_clean', 'Fecha_error', 'Expediente_Semefo', 'Procedencia', 'Procedencia_clean',
         'Procedencia_direccion', 'Procedencia_alcaldia', 'Numero_acta', 'Acta_procedencia', 'Diagnostico', 
         'Diagnostico_estandar', 'Diagnostico_extendido', 'Sexo', 'Edad', 'Edad_clean','Edad_error', 'Tipo_restos', 
         'Bitacora_ingresos', 'Pagina_PDF', 'Pagina_PDF_clean', 'Foja', 'Posible_identificado_como', 'Observaciones_capturista']]
df['Diagnostico'] = df['Diagnostico'].astype("category")

## Sexo

In [16]:
replace_dict = {'Masculino':'Masculino', 'Femenino': 'Femenino', 'S-D':'S-D',
                'M':'Masculino', 'Masculino ':'Masculino', 'M.F':'S-D', 'N':'S-D',
                'Mascullino':'Masculino', 'I':'S-D', 'S-D ':'S-D', 'MAsculino':'Masculino',
                'masculino':'Masculino', '  Masculino':'Masculino'}
df['Sexo'] = df['Sexo'].replace(replace_dict).astype("category")
df['Sexo'].value_counts()

Sexo
Masculino    73740
Femenino     20601
S-D           2519
Name: count, dtype: int64

## Edad

In [17]:
from math import floor
def replace_edades_menor_al_año(edad):
    if type(edad)==str:
        edad_ = simplify_string(edad)
        edad_ = edad_.replace(' 1/2','.5')
        edad_ = edad_.replace('horas','hora').replace('dias','dia').replace('semanas','semana').replace('meses','mes').replace('anos','ano')
        dic_time = {'ano':1, 'mes':12, 'semana':52, 'dia':365, 'hora':8766}
        for nombre, n in dic_time.items():
            if nombre in edad_:
                edad_ = simplify_string( edad_.replace(nombre,'') )
                try: edad_ = float(edad_)
                except ValueError: return edad
                return floor( edad_ / n )
        dic_time = {'a':1, 'm':12, 'd':365}
        for nombre, n in dic_time.items():
            if nombre in edad_:
                edad_ = simplify_string( edad_.replace(nombre,'') )
                try: edad_ = float(edad_)
                except ValueError: return edad
                return floor( edad_ / n )
        try: edad_ = float(edad_)
        except(ValueError): return edad
        return floor(edad_)
    return edad

In [18]:
dic_replace = {'[1-10 días]':0, '10 SEMNS':0, '15-30 días':0, '2 mese':0, '2 o 3 meses':0, '30 - 90 días':0, '5 meses 15 días':0, 
               '6 meses UI':0, '7 - 12 meses':0, '72 hrs':0, '-12 horas':1, '1 año 3 meses':1, '1 año 4 meses':1, '1 año 7 meses':1, 
               '1 Y 8 meses':1, '15 mese':1, '18 mese':1, '3 años y medio ':3, '7 añoss':7, '[8]':8, '-72 años':72, ' S-D':'S-D', 
               '-':'S-D', '?':'S-D', 'F':'S-D', 'M':'S-D', 'S-d':'S-D', '-72 años':72, '-1':1,
               ' +3-5':4,'7-9':8,' +6-8':7,' +7-9':8,'4-6':5,'5-7':6,'6-8':7,'4 y 5':'S-D',' +4-6':5,'8 años7':8,'más 72':'S-D',
               '1 - 3':2,'1-2':1,'9-9':9,'3-4':3,'7-8':7,'2-4':3,'8 Ins.':8,'41 n':41,'R.N':0,'18-25':21,'8 172':'S-D','7 - 9':8,
               '25-30':27,' +3-4':3,' +5-6':5,' +2-3':2,' +4-7':5,' +7-8':7,' +5-7':6,' +6-7':6,'25-40':'S-D','20 o 24':22,'25/30':22
              }

df['Edad_error'] = df['Edad_error'].apply(replace_edades_menor_al_año)
df['Edad_error'] = df['Edad_error'].replace(dic_replace)
df['Edad_error'] = pd.to_numeric(df['Edad_error'], errors='coerce')
df['Edad_error'].value_counts().to_dict()

df['Edad_clean'] = df['Edad_clean'].fillna(df['Edad_error']).drop_duplicates()
df['Edad_clean'] = df['Edad_clean'].abs().mask(df['Edad_clean']>99)


## Tipo_restos

In [19]:
replace_dict = {'Cadáver':'Cadáver', 'Cadáver ':'Cadáver', 'Feto':'Feto', 'Miembros':'Miembros',
                'Recién nacido':'Recién nacido', 'Miembros  ':'Miembros', 'Restos óseos':'Restos óseos',
                'S-D':'S-D', 'Cádaver':'Cadáver', 'Miembros ':'Miembros', 'Recién nacidos':'Recién nacido',
                'Cadáver. ':'Cadáver', 'Cadáver y Feto':'S-D'}

df['Tipo_restos'] = df['Tipo_restos'].replace(replace_dict).astype("category")
df['Tipo_restos'].value_counts()

Tipo_restos
Cadáver          91254
Feto              2843
Miembros          2062
Recién nacido      633
Restos óseos        48
S-D                 20
Name: count, dtype: int64

In [20]:
cat_nom = ['desconocido', 'feto', 'desconocida', 'r. nacido', 'recien nacido', 'r. nacida', 'recien nacida',
           's-d', 'r nacido', 'osamenta', 'restos humanos', 'r nacida', 'placenta', 'm.i. de:',
           'restos oseos', 'miembro inferior', 'fetos', 'm. inf.', 'm. inf. izq.', 'r-nacida',
           'restos placentarios', 'mi de', 'm s de desconocido', 'nacida', ]
data = df.loc[ df['Nombre_completo'].isin(cat_nom), ['ID','Nombre_completo','Tipo_restos']]
data

,ID,Nombre_completo,Tipo_restos
729,BO_1968_00730,desconocido,Cadáver
730,BO_1968_00731,desconocido,Cadáver
731,BO_1968_00732,desconocida,Cadáver
732,BO_1968_00733,desconocido,Cadáver
733,BO_1968_00734,desconocido,Cadáver
...,...,...,...
96815,BO_1982_07449,m s de desconocido,Miembros
96821,BO_1982_07455,m s de desconocido,Miembros
96839,BO_1982_07473,m s de desconocido,Miembros
96853,BO_1982_07487,restos placentarios,Miembros


In [21]:
cat_prob = [['restos humanos','Miembros'], ['m.i. de:','Miembros'], ['desconocido','Feto'], ['feto','Cadáver'], 
            ['feto','Recién nacido'], ['placenta','Cadáver'], ['restos humanos','Cadáver'],
            ['restos placentarios','Cadáver'], ['osamenta','Miembros']]
data = df[['ID','Nombre_completo','Tipo_restos']]

data = [data[(data['Nombre_completo']==nom) & (data['Tipo_restos']==tipo)] for nom,tipo in cat_prob ]
data = pd.concat(data)
data

,ID,Nombre_completo,Tipo_restos
8774,BO_1969_04056,restos humanos,Miembros
8931,BO_1969_04213,restos humanos,Miembros
9474,BO_1969_04756,restos humanos,Miembros
19466,BO_1971_04393,restos humanos,Miembros
19542,BO_1971_04469,restos humanos,Miembros
19576,BO_1971_04503,restos humanos,Miembros
19648,BO_1971_04575,restos humanos,Miembros
19666,BO_1971_04593,restos humanos,Miembros
19730,BO_1971_04657,restos humanos,Miembros
25103,BO_1972_04506,restos humanos,Miembros


## Bitacora_ingresos

In [22]:
df['Bitacora_ingresos'] = df['Bitacora_ingresos'].astype("category")

## Posible_identificado_como
## Observaciones_capturista

In [23]:
df['Observaciones_capturista'] = df['Observaciones_capturista'].replace('s-d',nan)
df['Posible_identificado_como'] = df['Posible_identificado_como'].replace('s-d',nan)
df['Posible_identificado_como'] = df['Posible_identificado_como'].apply(lambda s: 'Posible identificado como: '+s if type(s)==str else nan)
df['Observaciones'] = df[['Posible_identificado_como', 'Observaciones_capturista']].apply(lambda l: '.   '.join([s for s in l if type(s)==str]), axis=1 )
df['Observaciones'] = df['Observaciones'].replace({'':nan})
df[['Posible_identificado_como', 'Observaciones_capturista', 'Observaciones']]
df = df.drop(['Posible_identificado_como', 'Observaciones_capturista'], axis=1)
df.columns

Index(['ID', 'Numero_progresivo', 'Nombre_completo', 'Primer_apellido',
       'Segundo_apellido', 'Nombres_propios', 'Fecha', 'Fecha_clean',
       'Fecha_error', 'Expediente_Semefo', 'Procedencia', 'Procedencia_clean',
       'Procedencia_direccion', 'Procedencia_alcaldia', 'Numero_acta',
       'Acta_procedencia', 'Diagnostico', 'Diagnostico_estandar',
       'Diagnostico_extendido', 'Sexo', 'Edad', 'Edad_clean', 'Edad_error',
       'Tipo_restos', 'Bitacora_ingresos', 'Pagina_PDF', 'Pagina_PDF_clean',
       'Foja', 'Observaciones'],
      dtype='object')

## Desconocidos

1. S-D, desconocidos, feto, r nacido, etc en completo
2. desconocido o nan en parciales
3. solo un caracter en parciales
4. suma de casos especiales
5. miembros

In [24]:
cat_des = ['s-d', 'descono', 'feto', 'nacid', 'recien', 'osamenta', 'restos']
col_des = {s:df['Nombre_completo'].str.contains(s) for s in cat_des}
col_des = pd.concat(col_des,axis=1).any(axis=1)
df['Desconocido_ncompleto'] = col_des
col_des.value_counts()

False    78696
True     18164
Name: count, dtype: int64

In [25]:
# agrega buscar qie sea una sola letra y separa por columna
des_nom = df[['Primer_apellido', 'Segundo_apellido', 'Nombres_propios']]
des_nom = des_nom.replace({s:nan for s in ['s-d','desconocido', 'desconocida']})
des_nom = des_nom.applymap(simplify_string, str_case='lower', remove_special=True)
des_nom = des_nom.applymap(lambda s: len(s) if type(s)==str else 0)
des_nom = (des_nom > 1).sum(axis=1)
df['Desconocido_nseparado'] = des_nom
des_nom.value_counts(dropna=False)

3    73655
0    17598
2     4943
1      664
Name: count, dtype: int64

In [26]:
cat_mie = ['m. ', 'mi ', 'ms ', 'm i ', 'm s ', 'm.s.', 'm.i.', 'miembro', 'placenta']
col_mie = {s:df['Nombre_completo'].str.contains(s) for s in cat_mie}
col_mie = pd.concat(col_mie,axis=1).any(axis=1)
df['Desconocido_nmiembro'] = col_mie
col_mie.value_counts()

False    91745
True      5115
Name: count, dtype: int64

## Exportar y profile

In [27]:
df.to_csv('data_clean/SEMEFO_BO_all_revision.csv', index=False)
df.to_pickle('data_clean/SEMEFO_BO_all.pkl')

f_profile = 'profiles/SEMEFO_BO_all_revision_profile.html'
minimal_profile(df, f_profile, force_profile=True)


Export report to file: 100%|██████████████████████| 1/1 [00:00<00:00, 37.64it/s]


## Hipotesis

In [28]:
df.columns

Index(['ID', 'Numero_progresivo', 'Nombre_completo', 'Primer_apellido',
       'Segundo_apellido', 'Nombres_propios', 'Fecha', 'Fecha_clean',
       'Fecha_error', 'Expediente_Semefo', 'Procedencia', 'Procedencia_clean',
       'Procedencia_direccion', 'Procedencia_alcaldia', 'Numero_acta',
       'Acta_procedencia', 'Diagnostico', 'Diagnostico_estandar',
       'Diagnostico_extendido', 'Sexo', 'Edad', 'Edad_clean', 'Edad_error',
       'Tipo_restos', 'Bitacora_ingresos', 'Pagina_PDF', 'Pagina_PDF_clean',
       'Foja', 'Observaciones', 'Desconocido_ncompleto',
       'Desconocido_nseparado', 'Desconocido_nmiembro'],
      dtype='object')

In [29]:
data = df.loc[df['Expediente_Semefo']=='S-D', ['Nombre_completo', 'Fecha',  'Expediente_Semefo', 'Procedencia',  'Numero_acta']] \
         .replace('S-D',nan)
data

,Nombre_completo,Fecha,Expediente_Semefo,Procedencia,Numero_acta
613,cabrera perea evaristo,1968-08-28,NaN,NaN,NaN
662,chemor vda. de chuayffet,1968-10-31,NaN,NaN,NaN
711,chumasero bueno andres,1968-12-04,NaN,NaN,NaN
1416,figueroa de la luz sara,1968-08-13,NaN,NaN,NaN
2390,desconocido,1968-09-25,NaN,H.T.V.,74242
...,...,...,...,...,...
96410,desconocido,1982-11-18,NaN,13a,4591
96554,desconocido,NaN,NaN,NaN,NaN
96572,desconocido,NaN,NaN,NaN,NaN
96575,desconocido,1982-12-27,NaN,33a,2375


In [30]:
data[(data['Numero_acta'].notna()) & (data['Procedencia'].isna()) ]

,Nombre_completo,Fecha,Expediente_Semefo,Procedencia,Numero_acta
18422,m. sup. de gerardo villa,1971-01-08,NaN,NaN,73023
21504,desconocida,1972-01-11,NaN,NaN,67071
21918,desconocido,1972-06-01,NaN,NaN,71259
22474,diaz sanchez ana maria,NaN,NaN,NaN,31382
22593,feto,NaN,NaN,NaN,2
22645,feto,1972-03-30,NaN,NaN,31523
73988,msis de esteban caltzontzin,1979-06-22,NaN,NaN,1065


In [31]:
df.loc[df['Fecha_clean'].dt.year==1972, 'Expediente_Semefo']

20598       5
20599      49
20600      71
20601     101
20602     127
         ... 
26365    5039
26366    5674
26367    5770
29391     S-D
30280     229
Name: Expediente_Semefo, Length: 5754, dtype: object

In [32]:
df['Expediente_Semefo'].value_counts()

Expediente_Semefo
S-D       584
108        21
2861       20
319        20
25         20
         ... 
839-79      1
829-79      1
63097       1
382-79      1
92/83       1
Name: count, Length: 7795, dtype: int64

In [47]:
key_col = ['ID','Nombre_completo', 'Sexo', 'Edad_clean', 'Fecha_clean', 'Expediente_Semefo', 'Acta_procedencia', 'Diagnostico']

In [48]:
df.loc[df['Procedencia'].str.contains('SC'), key_col]

,ID,Nombre_completo,Sexo,Edad_clean,Fecha_clean,Expediente_Semefo,Acta_procedencia,Diagnostico
38887,BO_1975_00318,alatorre carlos,Masculino,NaN,NaT,S-D,SC : 4596,S-D
39665,BO_1975_01096,cardenas ismael roberto,Masculino,NaN,1975-12-08,S-D,SC : 7098,HsPSFPs
41578,BO_1975_03009,molinas de de la poza margarita,Femenino,NaN,1975-05-21,2573,SC-CH : 300,??
41996,BO_1975_03427,morales gonzalez j luis,Masculino,NaN,1975-12-18,S-D,SC : 7098-7,HsPAF
45432,BO_1976_00002,arbel d. viuda de ch. esperanza,Femenino,NaN,1976-01-03,25,SC : 0014 M-105,BN+CH
49081,BO_1976_03651,ochoa melendez jesus,Masculino,NaN,1976-08-02,4119,SC : 3322,HPAFPA
49669,BO_1976_04239,rojas zamora pablo,Masculino,NaN,1976-05-11,2539,SC : 2840,HPAFPC
52423,BO_1977_00086,alonso hernandez jose luis,Masculino,NaN,1977-03-29,1719,SC : 1903,HPAFPC
55665,BO_1977_03328,martinez magdaleno celia,Femenino,NaN,1977-09-03,4712,SC : 5430,HPAFPT
55666,BO_1977_03329,montoya ortiz cruz e.,Femenino,NaN,1977-09-03,4713,SC : 5430,HPAFPC


In [49]:
df.loc[df['Procedencia'].str.contains('PGR'), key_col]

,ID,Nombre_completo,Sexo,Edad_clean,Fecha_clean,Expediente_Semefo,Acta_procedencia,Diagnostico
35057,BO_1974_02219,iturralde arguelles joaquin,Masculino,NaN,1974-06-22,3020,PGR : 3032,HPAFPC
37372,BO_1974_04534,desconocido (1),Masculino,NaN,1974-02-15,791,PGR : 746,HsPPAEP
37373,BO_1974_04535,desconocido (2),Masculino,NaN,1974-02-15,792,PGR : 746,HPAFPC
37374,BO_1974_04536,desconocido (3),Masculino,NaN,1974-02-15,793,PGR : 746,HsPAFPsTA
37375,BO_1974_04537,desconocida (4),Femenino,NaN,1974-02-15,794,PGR : 746,HPAFPC
37376,BO_1974_04538,desconocida (5),Femenino,NaN,1974-02-15,795,PGR : 746,HsPAPPTyA
57086,BO_1977_04749,s. fambosky lucia,Femenino,NaN,1977-02-28,1137,PGR M-XIII,Dispensa


In [60]:
data = df.loc[df['Diagnostico_estandar'].str.contains('HPAF')==True, key_col]
index = data['Acta_procedencia'].value_counts()>2
index = index[index].index
data[data['Acta_procedencia'].isin(index)].sort_values(by='Acta_procedencia')

,ID,Nombre_completo,Sexo,Edad_clean,Fecha_clean,Expediente_Semefo,Acta_procedencia,Diagnostico
58540,BO_1977_06203,desconocido,Masculino,NaN,1977-06-25,3390,12a : 1512,HsPAFPs
58514,BO_1977_06177,desconocido,Masculino,NaN,1977-06-25,3390,12a : 1512,HsPAFPs
58506,BO_1977_06169,desconocido,Masculino,NaN,1977-06-25,3391,12a : 1512,HsPAFPsT
33721,BO_1974_00883,calixto de la cruz antonio a.,Masculino,NaN,1974-10-01,4715,15a : 1993,HsPAFPC
35851,BO_1974_03013,maldonado sanchez jose a,Masculino,NaN,1974-10-01,4716,15a : 1993,HsPAFPsCT
36381,BO_1974_03543,perales hernandez guillermo,Masculino,NaN,1974-10-02,4714,15a : 1993,HPAFPC
63752,BO_1978_04249,rodriguez de san miguel martha,Femenino,NaN,1978-03-12,1435,18a : 644,HPAFPT
59943,BO_1978_00440,bustos rodriguez jorge o.,Masculino,NaN,1978-03-12,1437,18a : 644,HPAFPC
59942,BO_1978_00439,bustos noriega jorge,Masculino,NaN,1978-03-12,1436,18a : 644,HPAFPT
70490,BO_1979_03833,ortiz de montano dolores,Masculino,NaN,1979-05-08,2008,19a : 634,HPAFPT


In [95]:
data = df.loc[df['Diagnostico_estandar'].str.contains('HPAF')==True, key_col]
data.groupby('Fecha_clean').filter(lambda g: len(g)>3).sort_values(by=['Fecha_clean','Acta_procedencia'])



,ID,Nombre_completo,Sexo,Edad_clean,Fecha_clean,Expediente_Semefo,Acta_procedencia,Diagnostico
35497,BO_1974_02659,mendoza pena antonio,Masculino,NaN,1974-02-15,780,23a : 81,HPAFPC
37373,BO_1974_04535,desconocido (2),Masculino,NaN,1974-02-15,792,PGR : 746,HPAFPC
37374,BO_1974_04536,desconocido (3),Masculino,NaN,1974-02-15,793,PGR : 746,HsPAFPsTA
37375,BO_1974_04537,desconocida (4),Femenino,NaN,1974-02-15,794,PGR : 746,HPAFPC
33909,BO_1974_01071,devesa sanabre angel,Masculino,NaN,1974-03-04,2147,13a : 2436,HPAFPC
...,...,...,...,...,...,...,...,...
70453,BO_1979_03796,nunez barrionuevo julio c.,Masculino,NaN,1979-12-25,7380,33a : 2602,HPAFP
67052,BO_1979_00395,ayala hernandez jose luis,Masculino,NaN,1979-12-26,7381,11a : 3393,HPAFPC
69611,BO_1979_02954,luna bonilla jose e,Masculino,NaN,1979-12-26,7376,13a : 2295,HPAFPT
71557,BO_1979_04900,ramirez oseguera carlos,Masculino,NaN,1979-12-26,7365,20a : 3693,HPAFPT


In [68]:
from datetime import datetime

date_min = datetime(1968, 10, 2)
data_max = datetime(1968, 10, 16)

df.loc[df['Fecha_clean'].between(date_min, data_max, inclusive='both'), key_col].sort_values(by='Fecha_clean')

In [78]:
import datetime

date_min = datetime.datetime(1975, 4, 25)
date_max = date_min + datetime.timedelta(days=7)

df.loc[df['Fecha_clean'].between(date_min, date_max, inclusive='both'), key_col].sort_values(by=['Fecha_clean','Acta_procedencia'])

,ID,Nombre_completo,Sexo,Edad_clean,Fecha_clean,Expediente_Semefo,Acta_procedencia,Diagnostico
42167,BO_1975_03598,orozco saenzs francisco,Masculino,NaN,1975-04-25,2129,AO-1 : 626,TM
44371,BO_1975_05802,desconocido,Masculino,NaN,1975-04-25,2117,AZ-1 : 78,TM
44372,BO_1975_05803,desconocido,Masculino,NaN,1975-04-25,2118,AZ-1 : 783,TM
44373,BO_1975_05804,desconocido,Masculino,NaN,1975-04-25,2119,AZ-1 : 783,TM
45312,BO_1975_06743,m inf de martin cruz,Masculino,NaN,1975-04-25,2127,BJ-CM : 797,AQxAgenos
...,...,...,...,...,...,...,...,...
42374,BO_1975_03805,paredes tapia ma guadalupe,Femenino,NaN,1975-05-02,2537,MH-HRL : 192,QuemEP
43337,BO_1975_04768,samano arriaga camilo,Masculino,NaN,1975-05-02,2546,MH-HRL : 438,HPAFPA
43803,BO_1975_05234,valladolid romero lidia,Femenino,NaN,1975-05-02,2251,TP-1 : 231,TCE
40291,BO_1975_01722,garcia montoya margarita,Femenino,NaN,1975-05-02,2268,VC-HB : 421,TCE
